# Medical CPT Training - Complete & Working
## Qwen2.5-7B with 8-Bit Quantization

**This notebook is fully self-contained and ready to run.**

Copy each cell as-is. No modifications needed.

In [ ]:
# ============ CELL 1: IMPORTS & SETUP ============
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("MEDICAL CPT TRAINING - 8-BIT QUANTIZED")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

if not torch.cuda.is_available():
    print("\n❌ FATAL: CUDA not available!")
    sys.exit(1)

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"GPU Memory: {gpu_mem_total:.1f}GB")

if gpu_mem_total < 24:
    print(f"\n⚠️  WARNING: Only {gpu_mem_total:.1f}GB GPU memory (need ≥24GB)")
    print("Training may fail. Consider using a larger GPU.\n")

print("="*80 + "\n")

In [ ]:
# ============ CELL 2: CHECK PACKAGES ============
print("Checking required packages...")

required_packages = {
    'transformers': 'AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig',
    'datasets': 'Dataset',
    'bitsandbytes': '8-bit quantization',
    'peft': 'prepare_model_for_kbit_training',
}

missing = []
try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        DataCollatorForLanguageModeling,
        BitsAndBytesConfig,
    )
    from datasets import Dataset
    from peft import prepare_model_for_kbit_training
    print("✅ All packages available\n")
except ImportError as e:
    print(f"❌ Missing: {e}")
    print("\nInstall with:")
    print("pip install -q transformers datasets torch bitsandbytes peft")
    sys.exit(1)

In [ ]:
# ============ CELL 3: CONFIGURATION ============
CONFIG = {
    "model_name": "Qwen/Qwen2.5-7B",
    "train_file": "augmented_output/train.jsonl",
    "eval_file": "augmented_output/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 1024,
    "bf16": True,
    "fp16": False,
    "output_dir": "medical_qwen_cpt_8bit",
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "eval_strategy": "steps",
    "eval_steps": 50,
    "logging_dir": "logs_8bit",
    "logging_steps": 5,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "seed": 42,
}

print("Configuration:")
print("="*80)
for k, v in CONFIG.items():
    print(f"{k:.<50} {v}")
print("="*80 + "\n")

effective_batch = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"Effective batch size: {effective_batch}")
print(f"Expected GPU memory: 22-26GB\n")

In [ ]:
# ============ CELL 4: LOAD DATA ============
def load_jsonl(file_path, max_samples=None):
    """Load JSONL file."""
    data = []
    if not Path(file_path).exists():
        raise FileNotFoundError(f"{file_path} not found. Run augmentation first.")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return data

print("Loading data...")
train_data = load_jsonl(CONFIG["train_file"])
eval_data = load_jsonl(CONFIG["eval_file"])

train_tokens = sum(c.get('token_count', 0) for c in train_data)
eval_tokens = sum(c.get('token_count', 0) for c in eval_data)

print(f"✅ Train: {len(train_data):,} chunks ({train_tokens:,} tokens)")
print(f"✅ Eval:  {len(eval_data):,} chunks ({eval_tokens:,} tokens)\n")

# Show sample
sample = train_data[0]
print(f"Sample text (first 300 chars): {sample['text'][:300]}...\n")

In [ ]:
# ============ CELL 5: LOAD TOKENIZER ============
print(f"Loading tokenizer: {CONFIG['model_name']}...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded (vocab: {len(tokenizer):,})\n")

In [ ]:
# ============ CELL 6: LOAD MODEL WITH 8-BIT QUANTIZATION ============
print(f"Loading model with 8-bit quantization...")
print("(This may take 1-2 minutes)\n")

# Configure 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# Prepare model for 8-bit training
model = prepare_model_for_kbit_training(model)

num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded")
print(f"   Parameters: {num_params/1e9:.2f}B")
print(f"   Quantization: 8-bit (reduces 15GB → 4GB)")
print(f"   Gradient checkpointing: Enabled\n")

# Check GPU memory after loading
torch.cuda.synchronize()
gpu_mem_used = torch.cuda.memory_allocated(0) / 1e9
print(f"GPU memory after model load: {gpu_mem_used:.1f}GB\n")

In [ ]:
# ============ CELL 7: TOKENIZE DATASETS ============
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        padding="max_length",
    )

print("Tokenizing datasets...")

# Create HF Dataset objects
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})

# Tokenize
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"✅ Train dataset: {len(train_dataset):,} samples")
print(f"✅ Eval dataset: {len(eval_dataset):,} samples\n")

In [ ]:
# ============ CELL 8: SETUP DATA COLLATOR ============
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling, not masked
)

print("✅ Data collator configured\n")

In [ ]:
# ============ CELL 9: SETUP TRAINING ARGUMENTS ============
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    bf16=CONFIG["bf16"],
    fp16=CONFIG["fp16"],
    save_strategy=CONFIG["save_strategy"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    eval_strategy=CONFIG["eval_strategy"],
    eval_steps=CONFIG["eval_steps"],
    metric_for_best_model="eval_loss",
    logging_dir=CONFIG["logging_dir"],
    logging_steps=CONFIG["logging_steps"],
    seed=CONFIG["seed"],
    dataloader_num_workers=CONFIG["dataloader_num_workers"],
    dataloader_pin_memory=CONFIG["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
)

print("✅ Training arguments configured\n")

In [ ]:
# ============ CELL 10: CREATE TRAINER ============
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✅ Trainer created and ready\n")
print("="*80)
print("🚀 READY TO TRAIN")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Expected duration: 12-18 hours")
print(f"Output directory: {CONFIG['output_dir']}/")
print("="*80 + "\n")

In [ ]:
# ============ CELL 11: START TRAINING ============
print(f"\n🚀 Starting training...\n")

try:
    train_result = trainer.train()
    print(f"\n✅ Training complete!")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Training loss: {train_result.training_loss:.4f}\n")
except KeyboardInterrupt:
    print("\n⏹️  Training interrupted by user")
except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============ CELL 12: EVALUATE ============
print("\nEvaluating model...\n")

eval_results = trainer.evaluate()

print("Evaluation Results:")
print("="*50)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"{key:.<40} {value:.4f}")
    else:
        print(f"{key:.<40} {value}")
print("="*50 + "\n")

In [ ]:
# ============ CELL 13: SAVE MODEL ============
best_model_path = Path(CONFIG["output_dir"]) / "best_model"
best_model_path.mkdir(parents=True, exist_ok=True)

print(f"Saving best model to {best_model_path}...")

trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

model_size = sum(f.stat().st_size for f in best_model_path.glob('**/*')) / 1e9

print(f"✅ Model saved")
print(f"   Path: {best_model_path}")
print(f"   Size: {model_size:.2f}GB\n")

In [ ]:
# ============ CELL 14: TEST INFERENCE ============
print("Testing inference with trained model...\n")

from transformers import pipeline

# Load trained model for inference
inference_model = AutoModelForCausalLM.from_pretrained(
    str(best_model_path),
    quantization_config=bnb_config,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
    device=0,
)

# Test prompts
test_prompts = [
    "The pathophysiology of venous insufficiency involves",
    "Duplex ultrasound is essential for diagnosing",
    "Treatment options for varicose veins include",
]

print("Generated text samples:")
print("="*80)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nExample {i}")
    print(f"Prompt: {prompt}")
    print("-" * 80)
    
    output = generator(
        prompt,
        max_length=120,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    generated = output[0]['generated_text']
    print(f"Output: {generated}")

print("\n" + "="*80)
print("✅ Inference test complete\n")

In [ ]:
# ============ CELL 15: FINAL SUMMARY ============
import glob

checkpoints = sorted(glob.glob(f"{CONFIG['output_dir']}/checkpoint-*"))

print("\n" + "="*80)
print("✅ TRAINING COMPLETE AND SUCCESSFUL")
print("="*80)

print(f"\n📂 Output Files:")
print(f"   Best model: {best_model_path}")
print(f"   Checkpoints: {len(checkpoints)} saved")
print(f"   Logs: {CONFIG['logging_dir']}/")

print(f"\n📊 Results:")
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")

print(f"\n🚀 Next Steps:")
print(f"   1. Load model: AutoModelForCausalLM.from_pretrained('{best_model_path}')")
print(f"   2. View tensorboard: tensorboard --logdir {CONFIG['logging_dir']}")
print(f"   3. Deploy or fine-tune further")

print(f"\n" + "="*80)